# Data cleaning

In [ ]:
import pandas as pd
import numpy as np

# --- STEP 0: LOAD DATA ---
from google.colab import drive
drive.mount('/content/drive')

# Path to the Excel file
path = '/content/drive/My Drive/Datathon Dataset.xlsx'

# Read main data from the 'Data - Main' sheet
df_main = pd.read_excel(path, sheet_name='Data - Main')

# Read linkage data from the 'Others - Category Linkage' sheet
df_linkage = pd.read_excel(path, sheet_name='Others - Category Linkage')

df_main.info()
# --- DROP EMPTY COLUMNS ---
cols_to_drop = ['Cost Ctr', 'WBS element', 'Ref.key (header) 1', 'Clrng doc.']
df_main = df_main.drop(columns=cols_to_drop, errors='ignore')
print(f"Dropped empty columns: {cols_to_drop}")

# --- DROP SPECIFIC COLUMNS BY INDEX (corresponding to names) ---
columns_to_drop_by_name = ['Amount in doc. curr.', 'Amt in loc.cur.', 'Curr.', 'LCurr', 'Rate (USD)']
df_main = df_main.drop(columns=columns_to_drop_by_name, errors='ignore')
print(f"Dropped columns by index (name): {columns_to_drop_by_name}")

#to understand dataset structure
df_main.info()

# --- CUSTOM CLEANING: UPDATE 'Non Netting AP' category and index ---
df_main['Category'] = df_main['Category'].replace({'Non Netting AP': 'Non-Netting AP'})
df_main.loc[df_main['Category'] == 'Non-Netting AP', 'Category Index'] = 6
print("Updated 'Non Netting AP' to 'Non-Netting AP' and set 'Category Index' to 6.")


Check duplicated row

In [ ]:
#to check duplicated row
df_main.duplicated().sum()

Anomaly flagging & Create Cashflow Bucket

In [ ]:
# --- STEP 4: ANOMALY FLAGGING ---

# 1. Convert Date to Datetime (Crucial for Weekend Check)
df_main['Pstng Date'] = pd.to_datetime(df_main['Pstng Date'])

# 2. Flag Weekend Transactions (Saturday & Sunday)
# Note: 5=Saturday, 6=Sunday
df_main['is_weekend'] = df_main['Pstng Date'].dt.dayofweek >= 5

# 3. Flag System Duplicates ("Super Strict" Logic)
# We mark True if Name, Date, Amount, Vendor, Reference, AND Assignment are identical
# keep='first' means the first occurrence is False (valid), subsequent ones are True (duplicates)
duplicate_cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
df_main['is_duplicate'] = df_main.duplicated(subset=duplicate_cols, keep='first')

# 4. Flag Statistical Outliers (Z-Score > 3)
# We calculate Z-score within each 'Category' to respect different scales
df_main['z_score'] = df_main.groupby('Category')['Amount in USD'].transform(
    lambda x: (x - x.mean()) / x.std()
)
df_main['is_outlier'] = np.abs(df_main['z_score']) > 3

# Clean up temporary Z-score column if you don't need it
df_main = df_main.drop(columns=['z_score'])

# --- STEP 5: CREATE CASHFLOW BUCKET ---
financing_categories = [12, 13, 14, 15, 19, 20, 21, 24]
df_main['cashflow_Bucket'] = 'Operating'
df_main.loc[df_main['Category Index'].isin(financing_categories), 'cashflow_Bucket'] = 'Financing'
print(f"Created 'cashflow_Bucket' column based on 'Category Index'")

# --- SUMMARY OF RESULTS ---
print("\n--- Step 4 Anomaly Summary ---")
print(f"Total Rows: {len(df_main)}")
print(f"Weekend Transactions: {df_main['is_weekend'].sum()} (Pattern - Keep for Forecast)")
print(f"Statistical Outliers: {df_main['is_outlier'].sum()} (Spikes - Review)")
print(f"System Duplicates: {df_main['is_duplicate'].sum()} (Errors - Exclude)")

# Show sample of the new flags
print("\nSample Data with Flags and Cashflow Bucket:")
print(df_main[['Pstng Date', 'Category', 'Category Index', 'Amount in USD', 'is_weekend', 'is_duplicate', 'is_outlier', 'cashflow_Bucket']].head())

# --- SAVE TO NEW CSV ---
df_main.to_csv('cleaned_data_with_cashflow.csv', index=False)
print("Saved updated data to 'cleaned_data_with_cashflow.csv'")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. 读取数据 (Read Excel Sheets)
# ==========================================

# 假设你的 Excel 文件名是这个 (如果不同，请修改这里)
excel_filename = 'Datathon Dataset.xlsx'

# 检查文件是否存在
if os.path.exists(excel_filename):
    print(f"✅ 找到文件: {excel_filename}")

    # 读取 'Data - Cash Balance' Sheet
    # 注意：sheet_name 参数必须完全匹配 Excel 里的名字，如果不确定可以先打印 sheet_names
    xls = pd.ExcelFile(excel_filename)
    print(f"Excel 里的 Sheet 列表: {xls.sheet_names}")

    # 自动寻找包含 'Balance' 和 'Main' 的 Sheet 名字，防止手打出错
    sheet_balance_name = [s for s in xls.sheet_names if 'Balance' in s][0]
    sheet_main_name = [s for s in xls.sheet_names if 'Main' in s][0]

    print(f"正在读取 Sheet: {sheet_balance_name} 和 {sheet_main_name}...")

    df_balance = pd.read_excel(excel_filename, sheet_name=sheet_balance_name)
    df_main = pd.read_excel(excel_filename, sheet_name=sheet_main_name)

    print("✅ 数据读取成功！")

else:
    print(f"❌ 错误：找不到文件 {excel_filename}。请确认文件已上传且名字正确。")
    # 如果找不到，停止运行后续代码
    raise FileNotFoundError(f"File {excel_filename} not found")

# ==========================================
# 2. 数据清洗与计算 (Data Processing)
# ==========================================

# 建立初始余额字典: { 'TW10': 1026662.22, ... }
# 注意：这里我们用 'Carryforward Balance (USD)' 这一列
initial_balances = dict(zip(df_balance['Name'], df_balance['Carryforward Balance (USD)']))

# 确保日期格式正确
df_main['Pstng Date'] = pd.to_datetime(df_main['Pstng Date'])
# 确保金额是数字
df_main['Amount in USD'] = pd.to_numeric(df_main['Amount in USD'], errors='coerce')

# 按周 (Weekly) 汇总 Net Cash Flow
# 使用 'W-MON' (每周一开始)
df_main['Week_Start'] = df_main['Pstng Date'].dt.to_period('W').apply(lambda r: r.start_time)

# 分组求和
weekly_data = df_main.groupby(['Name', 'Week_Start'])['Amount in USD'].sum().reset_index()
weekly_data.rename(columns={'Amount in USD': 'Net_Cash_Flow'}, inplace=True)

# ==========================================
# 3. 计算期末余额 (Calculate Ending Balance)
# ==========================================

def get_ending_balance(group):
    entity_name = group.name
    # 获取该 Entity 的初始余额，如果没有则默认为 0
    start_bal = initial_balances.get(entity_name, 0)

    # 按时间排序
    group = group.sort_values('Week_Start')

    # 核心公式: 当周余额 = 初始余额 + 累计现金流
    group['Ending_Balance'] = start_bal + group['Net_Cash_Flow'].cumsum()
    return group

# 应用函数
weekly_balance_df = weekly_data.groupby('Name').apply(get_ending_balance).reset_index(drop=True)

# ==========================================
# 4. 画图 (Visualisation)
# ==========================================

# 只选最具代表性的两个 Entity
target_entities = ['TW10', 'SS10']
plot_data = weekly_balance_df[weekly_balance_df['Name'].isin(target_entities)]

plt.figure(figsize=(12, 6))
# 绘制折线图
sns.lineplot(data=plot_data, x='Week_Start', y='Ending_Balance', hue='Name', linewidth=2.5)

# 添加 0 线 (Insolvency Line)
plt.axhline(0, color='red', linestyle='--', label='Zero Balance (Risk Line)')

# 美化图表
plt.title('Liquidity Analysis: Volatile (TW10) vs Stable (SS10)', fontsize=14, fontweight='bold', color='#830051')
plt.ylabel('Ending Cash Balance (USD)')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.legend()

# 保存并展示
plt.savefig('Ending_Balance_Comparison.png')
plt.show()

# 打印最后几周的数据，供你检查
print("\n🔍 最后 3 周的数据预览:")
print(plot_data.groupby('Name').tail(3)[['Name', 'Week_Start', 'Net_Cash_Flow', 'Ending_Balance']])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile

# ==========================================
# 1. Load Data (Robust File Check)
# ==========================================
excel_filename = 'Datathon Dataset.xlsx'

if not os.path.exists(excel_filename):
    print(f"❌ Error: '{excel_filename}' not found. Please upload the file.")
else:
    print(f"✅ Found '{excel_filename}'. Reading data...")
    xls = pd.ExcelFile(excel_filename)

    # Auto-detect sheet names to avoid errors
    sheet_balance = [s for s in xls.sheet_names if 'Balance' in s][0]
    sheet_main = [s for s in xls.sheet_names if 'Main' in s][0]

    df_balance = pd.read_excel(excel_filename, sheet_name=sheet_balance)
    df_main = pd.read_excel(excel_filename, sheet_name=sheet_main_name)
    print("✅ Data loaded successfully!")

# ==========================================
# 2. Process Data (Calculate Ending Balance)
# ==========================================
# Create a dictionary for Initial Balances (USD)
initial_balances = dict(zip(df_balance['Name'], df_balance['Carryforward Balance (USD)']))

# Format Dates and Numerics
df_main['Pstng Date'] = pd.to_datetime(df_main['Pstng Date'])
df_main['Amount in USD'] = pd.to_numeric(df_main['Amount in USD'], errors='coerce')

# Aggregate Net Cash Flow by Week
df_main['Week_Start'] = df_main['Pstng Date'].dt.to_period('W').apply(lambda r: r.start_time)
weekly_data = df_main.groupby(['Name', 'Week_Start'])['Amount in USD'].sum().reset_index()
weekly_data.rename(columns={'Amount in USD': 'Net_Cash_Flow'}, inplace=True)

# Function to calculate cumulative balance
def calculate_ending_balance(group):
    entity_name = group.name
    start_bal = initial_balances.get(entity_name, 0)
    group = group.sort_values('Week_Start')
    # Formula: Ending Balance = Initial + Cumulative Sum of Flows
    group['Ending_Balance'] = start_bal + group['Net_Cash_Flow'].cumsum()
    return group

# Apply calculation
df_final_balance = weekly_data.groupby('Name').apply(calculate_ending_balance).reset_index(drop=True)

# ==========================================
# 3. Generate 8 Charts (English)
# ==========================================
entities = df_final_balance['Name'].unique()
generated_files = []

# Set Style
sns.set_theme(style="whitegrid")

for entity in entities:
    plt.figure(figsize=(10, 6))

    # Filter data for current entity
    subset = df_final_balance[df_final_balance['Name'] == entity]

    # Plot Line (Using AstraZeneca Mulberry Color)
    sns.lineplot(data=subset, x='Week_Start', y='Ending_Balance', color='#830051', linewidth=2.5)

    # Add Zero Line (Risk Threshold)
    plt.axhline(0, color='red', linestyle='--', linewidth=2, label='Insolvency Threshold (0 USD)')

    # Highlight Risk Area (Below 0)
    plt.fill_between(subset['Week_Start'], subset['Ending_Balance'], 0,
                     where=(subset['Ending_Balance'] < 0),
                     color='red', alpha=0.15, label='Liquidity Deficit')

    # Titles and Labels in English
    plt.title(f'Projected Liquidity Position: {entity}', fontsize=14, fontweight='bold', color='#333333')
    plt.xlabel('Timeline (Weekly)', fontsize=12)
    plt.ylabel('Ending Cash Balance (USD)', fontsize=12)
    plt.legend(loc='upper left')

    # Save Chart
    filename = f"{entity}_Liquidity_Chart.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    generated_files.append(filename)
    plt.close()
    print(f"Generated chart: {filename}")

# ==========================================
# 4. Zip Files for Download
# ==========================================
zip_name = "Liquidity_Charts.zip"
with zipfile.ZipFile(zip_name, 'w') as zipf:
    for file in generated_files:
        zipf.write(file)

print(f"\n🎉 SUCCESS! Download your file here: {zip_name}")
print("Go to the 'Files' folder on the left sidebar to download it.")

Cash Balance

# EDA


6 Check

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
import os # Import os for file path checks

# Define the path for the cleaned data file
cleaned_file_path = 'cleaned_data_with_cashflow.csv'

# --- Robust Data Loading and Preprocessing (if cleaned file doesn't exist) ---
if not os.path.exists(cleaned_file_path):
    print(f"'{cleaned_file_path}' not found. Performing initial data loading and cleaning...")
    from google.colab import drive

    # Check if drive is already mounted to avoid duplicate messages
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Prioritize mounted drive path, fallback to local path if not found
    excel_path_drive = '/content/drive/My Drive/Datathon Dataset.xlsx'
    excel_path_local = '/content/Datathon Dataset.xlsx'

    if os.path.exists(excel_path_drive):
        path = excel_path_drive
    elif os.path.exists(excel_path_local):
        path = excel_path_local
    else:
        raise FileNotFoundError(f"Datathon Dataset.xlsx not found at {excel_path_drive} or {excel_path_local}")

    # Read main data from the 'Data - Main' sheet
    df_main_init = pd.read_excel(path, sheet_name='Data - Main')

    # --- DROP EMPTY COLUMNS --- (from D5uhZJbaVVox)
    cols_to_drop = ['Cost Ctr', 'WBS element', 'Ref.key (header) 1', 'Clrng doc.']
    df_main_init = df_main_init.drop(columns=cols_to_drop, errors='ignore')

    # --- DROP SPECIFIC COLUMNS BY INDEX (corresponding to names) --- (from D5uhZJbaVVox)
    columns_to_drop_by_name = ['Amount in doc. curr.', 'Amt in loc.cur.', 'Curr.', 'LCurr', 'Rate (USD)']
    df_main_init = df_main_init.drop(columns=columns_to_drop_by_name, errors='ignore')

    # --- CUSTOM CLEANING: UPDATE 'Non Netting AP' category and index --- (from D5uhZJbaVVox)
    df_main_init['Category'] = df_main_init['Category'].replace({'Non Netting AP': 'Non-Netting AP'})
    df_main_init.loc[df_main_init['Category'] == 'Non-Netting AP', 'Category Index'] = 6

    # --- ANOMALY FLAGGING & CASHFLOW BUCKET (from HNBiu2DO0gVu) ---
    df_main_init['Pstng Date'] = pd.to_datetime(df_main_init['Pstng Date'])
    df_main_init['is_weekend'] = df_main_init['Pstng Date'].dt.dayofweek >= 5
    duplicate_cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_main_init['is_duplicate'] = df_main_init.duplicated(subset=duplicate_cols, keep='first')
    df_main_init['z_score'] = df_main_init.groupby('Category')['Amount in USD'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    df_main_init['is_outlier'] = np.abs(df_main_init['z_score']) > 3
    df_main_init = df_main_init.drop(columns=['z_score'])
    financing_categories = [12, 13, 14, 15, 19, 20, 21, 24]
    df_main_init['cashflow_Bucket'] = 'Operating'
    df_main_init.loc[df_main_init['Category Index'].isin(financing_categories), 'cashflow_Bucket'] = 'Financing'

    # Save the cleaned data for future use
    df_main_init.to_csv(cleaned_file_path, index=False)
    print(f"Successfully created and saved '{cleaned_file_path}'.")

# 1. Load the Preprocessed Data
df = pd.read_csv(cleaned_file_path, low_memory=False)

# Convert dates to datetime objects
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

# Define AZ Branding Colors (Purple, Lime Green, Magenta)
az_colors = ['#830051', '#C4D600', '#F2F2F2']
sns.set_palette(sns.color_palette(az_colors))

print("Data loaded successfully! Starting EDA Analysis...")

# --- Professional Note: Column Alias ---
# 如果你的列名是 'Amount in USD', 我们这里统一映射一下方便后续计算
if 'Amount in USD' in df.columns:
    df['Amount_USD'] = df['Amount in USD']

# 2. Define Financial Activity Mapping
def map_activity(idx):
    operating = [1.0, 2.0, 7.0, 8.0, 9.0, 10.0, 11.0, 16.0, 22.0, 25.0]
    financing = [12.0, 13.0, 14.0, 19.0, 23.0]
    if idx in operating: return 'Operating'
    elif idx in financing: return 'Financing'
    else: return 'Investing/Other'

df['Financial_Activity'] = df['Category Index'].apply(map_activity)
# ==========================================
# EDA 1: Volatility Analysis (Risk Assessment)
# ==========================================
# Calculate Standard Deviation ($ \sigma $) per Legal Entity
volatility = df.groupby('Name')['Amount in USD'].std().sort_values(ascending=False).reset_index()
volatility.columns = ['Entity', 'Std_Dev_USD']

plt.figure(figsize=(12, 6))
sns.barplot(data=volatility, x='Entity', y='Std_Dev_USD', palette='magma')
plt.title('Cash Flow Volatility per Legal Entity (Standard Deviation)')
plt.ylabel('Volatility (USD)')
plt.xlabel('Legal Entity')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("\n--- Insight: Volatility ---")
print(f"The most volatile entity is {volatility.iloc[0]['Entity']} with a Std Dev of {volatility.iloc[0]['Std_Dev_USD']:.2f} USD.")

weekly_data = df.groupby(['Name', pd.Grouper(key='Pstng Date', freq='W-MON')])['Amount_USD'].sum().reset_index()
volatility = weekly_data.groupby('Name')['Amount_USD'].std().sort_values(ascending=False).reset_index()
volatility.columns = ['Entity', 'Weekly_Volatility_StdDev']

print("--- Entity Volatility Ranking (Risk Metric) ---")
print(volatility)

plt.figure(figsize=(10, 5))
sns.barplot(data=volatility, x='Entity', y='Weekly_Volatility_StdDev', palette='magma')
plt.title('Weekly Cash Flow Volatility per Legal Entity')
plt.show()
# ==========================================
# EDA 2: Cash Flow Drivers (Operating/Investing/Financing)
# ==========================================
# Group by your newly created 'cashflow_Bucket'
bucket_summary = df.groupby('cashflow_Bucket')['Amount in USD'].apply(lambda x: x.abs().sum()).reset_index()

plt.figure(figsize=(8, 8))
plt.pie(bucket_summary['Amount in USD'], labels=bucket_summary['cashflow_Bucket'], autopct='%1.1f%%',
        startangle=140, colors=['#830051', '#C4D600', '#D0D0D0'])
plt.title('Distribution of Absolute Cash Flow by Bucket')
plt.show()

# ==========================================
# EDA 3: Detailed Category Drivers (Top 5 Outflows)
# ==========================================
# Identify Inflows vs Outflows based on sign
outflows = df[df['Amount in USD'] < 0].groupby('Category')['Amount in USD'].sum().abs().nlargest(5).reset_index()
inflows = df[df['Amount in USD'] > 0].groupby('Category')['Amount in USD'].sum().nlargest(5).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=outflows, x='Amount in USD', y='Category', ax=ax[0], color='#830051')
ax[0].set_title('Top 5 Outflow Drivers (Categories)')

sns.barplot(data=inflows, x='Amount in USD', y='Category', ax=ax[1], color='#C4D600')
ax[1].set_title('Top 5 Inflow Drivers (Categories)')

plt.tight_layout()
plt.show()

# ==========================================
# EDA 4: Anomaly Summary (The Auditor View)
# ==========================================
# Use the boolean flags from your preprocessing
anomaly_counts = pd.DataFrame({
    'Type': ['Duplicate', 'Outlier'],
    'Count': [ df['is_duplicate'].sum(), df['is_outlier'].sum()]
})

plt.figure(figsize=(10, 5))
sns.barplot(data=anomaly_counts, x='Type', y='Count', palette='Reds_d', width=0.3) # Added width=0.3 for slimmer bars
plt.title('Summary of Detected Anomalies')
plt.ylabel('Number of Transactions')
plt.show()

# ==========================================
# EDA 5: Seasonality Check (Monthly Trend)
# ==========================================
df['YearMonth'] = df['Pstng Date'].dt.to_period('M').astype(str)
monthly_trend = df.groupby('YearMonth')['Amount in USD'].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(monthly_trend['YearMonth'], monthly_trend['Amount in USD'], marker='o', color='#830051', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('Monthly Net Cash Flow Trend')
plt.xlabel('Month')
plt.ylabel('Net Cash Flow (USD)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# EDA 6: Payroll Focus (Structural Seasonality)
# ==========================================
payroll = df[df['Category'] == 'Payroll'].copy()
if not payroll.empty:
    payroll_daily = payroll.groupby('Pstng Date')['Amount in USD'].sum().abs().reset_index()
    plt.figure(figsize=(14, 4))
    plt.scatter(payroll_daily['Pstng Date'], payroll_daily['Amount in USD'], color='#830051', alpha=0.5, s=10)
    plt.title('Payroll Payment Cycles (Visualizing Month-End Spikes)')
    plt.ylabel('USD Amount')
    plt.show()

# 1. Load Data (for decomposition, also using cleaned_file_path)
df_decomp = pd.read_csv(cleaned_file_path, low_memory=False)
df_decomp['Pstng Date'] = pd.to_datetime(df_decomp['Pstng Date'])

# 2. Weekly Aggregation
# We aggregate by week (W-MON) to see the 'Business Rhythm'
df_weekly = df_decomp.groupby(pd.Grouper(key='Pstng Date', freq='W-MON'))['Amount in USD'].sum().reset_index()
df_weekly = df_weekly.set_index('Pstng Date')

# 3. Time Series Decomposition
# We use additive model: Observed = Trend + Seasonality + Noise
# Period=4 reflects a monthly cycle in weekly data
decomposition = seasonal_decompose(df_weekly['Amount in USD'], model='additive', period=4)

# 4. Visualization
plt.figure(figsize=(12, 10))

# Subplot 1: Observed (Original)
plt.subplot(411)
plt.plot(decomposition.observed, color='#830051', label='Original (Observed)')
plt.legend(loc='upper left')
plt.title('Weekly Net Cash Flow Decomposition')

# Subplot 2: Trend
plt.subplot(412)
plt.plot(decomposition.trend, color='#C4D600', label='Trend (Long-term direction)')
plt.legend(loc='upper left')

# Subplot 3: Seasonality
plt.subplot(413)
plt.plot(decomposition.seasonal, color='blue', label='Seasonality (Monthly Pattern)')
plt.legend(loc='upper left')

# Subplot 4: Noise (Residuals)
plt.subplot(414)
plt.scatter(decomposition.resid.index, decomposition.resid, color='red', s=10, label='Noise (Anomalies / Residuals)')
plt.axhline(0, color='black', linestyle='--')
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

# --- Analytical Insight ---
noise_std = decomposition.resid.std()
print(f"Analysis Summary:")
print(f"- Monthly Seasonality Peak: ${decomposition.seasonal.max():,.2f}")
print(f"- Unexplained Noise (Std Dev): ${noise_std:,.2f}")

# Load Data (for weekly aggregation, also using cleaned_file_path)
df_weekly_agg = pd.read_csv(cleaned_file_path, low_memory=False)
df_weekly_agg['Pstng Date'] = pd.to_datetime(df_weekly_agg['Pstng Date'])
amount_col = 'Amount in USD'  # 你的金额列名

# 3. Weekly Aggregation (The Core Logic)
# 按周 (W-MON: 每周一为起始) 和 实体 (Name) 汇总金额
# 这就是 "Weekly Net Cash Flow"
weekly_df = df_weekly_agg.groupby(['Name', pd.Grouper(key='Pstng Date', freq='W-MON')])[amount_col].sum().reset_index()
weekly_df.columns = ['Entity', 'Week_Start', 'Net_Cash_Flow_USD']

# 计算全公司的总周现金流 (Total Global Cash Flow)
global_weekly = weekly_df.groupby('Week_Start')['Net_Cash_Flow_USD'].sum().reset_index()

# ==========================================
# OUTPUT: Save the processed weekly data
# ==========================================
weekly_df.to_csv('weekly_net_cash_flow.csv', index=False)
print("SUCCESS: Weekly aggregated data saved as 'weekly_net_cash_flow.csv'")

# Show the first few rows of the result
print("\nSample of Weekly Data:")
print(weekly_df.head())

# Visualization 3: Cash Flow Composition by Bucket
fig_bucket = px.histogram(df, x='cashflow_Bucket', y='Amount in USD', color='cashflow_Bucket',
                          title='<b>Total Cash Flow by Activity (Operating vs Investing)</b>',
                          text_auto='.2s',
                          color_discrete_sequence=['#830051', '#C4D600', '#003865']) # AZ Brand Colors
fig_bucket.update_layout(template='plotly_white')
fig_bucket.show()

# ---------------------------------------------------------
# 1. SETUP & DATA LOADING (for volatility, also using cleaned_file_path)
# ---------------------------------------------------------
# Load the cleaned dataset
df_vol = pd.read_csv(cleaned_file_path) # Ensure this matches your upload

# Convert Dates to Datetime
df_vol['Pstng Date'] = pd.to_datetime(df_vol['Pstng Date'])
df_vol['Doc..Date'] = pd.to_datetime(df_vol['Doc..Date'])

# Create Time Features
df_vol['Week_Start'] = df_vol['Pstng Date'] - pd.to_timedelta(df_vol['Pstng Date'].dt.weekday, unit='D')
df_vol['Month'] = df_vol['Pstng Date'].dt.to_period('M').astype(str)

print("✅ Data Loaded Successfully!")
print(f"   Total Transactions: {len(df_vol)}")
print(f"   Date Range: {df_vol['Pstng Date'].min().date()} to {df_vol['Pstng Date'].max().date()}")
# Visualization 4: Top Categories Driving Volatility (The "Drivers")
# Calculate Variance (Volatility) per Category
cat_volatility = df_vol.groupby('Category')['Amount in USD'].std().sort_values(ascending=False).head(10).reset_index()

fig_vol = px.bar(cat_volatility, x='Amount in USD', y='Category', orientation='h',
                 title='<b>Top 10 Most Volatile Categories (Primary Risk Drivers)</b>',
                 labels={'Amount in USD': 'Standard Deviation (Volatility)'},
                 color='Amount in USD', color_continuous_scale='Reds')
fig_vol.update_layout(template='plotly_white', yaxis={'categoryorder':'total ascending'})
fig_vol.show()

print("\n✅ EDA COMPLETE. Use these charts to justify your Model Choice and Risk Strategy.")

# ==========================================
# 0. DATA PREPARATION (for weekend anomaly, also using cleaned_file_path)
# ==========================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set global style preferences for a cleaner look
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# Load Data
df_weekend = pd.read_csv(cleaned_file_path, low_memory=False)
weekend_df = df_weekend[df_weekend['is_weekend'] == True].copy()

# Calculate Frequencies
weekend_df['Ref_Freq'] = weekend_df.groupby('Reference')['Reference'].transform('count')
weekend_df['Assign_Freq'] = weekend_df.groupby('Assignment')['Assignment'].transform('count')

# Count the specific anomalies
# Count 1: Critical Isolated (Manual)
isolated_count = len(weekend_df[(weekend_df['Ref_Freq'] == 1) & (weekend_df['Assign_Freq'] == 1)])
# Count 2: Suspicious Missing Ref
missing_ref_count = len(weekend_df[weekend_df['Reference'].isna() | (weekend_df['Reference'].astype(str).str.strip() == '')])

# Prepare data for plotting
plot_data = pd.DataFrame({
    'Anomaly Type': ['Critical: Isolated Manual Entry\n(High Risk)', 'Warning: Missing Bank Ref\n(Audit Gap)'],
    'Count': [isolated_count, missing_ref_count],
    'Color_Code': ['#D32F2F', '#FBC02D'] # Deep Red for Critical, Amber for Warning
})

print(f"Data Ready: Isolated={isolated_count}, MissingRef={missing_ref_count}")

# ==========================================
# OPTION 1: Modern Minimalist Bar Chart
# ==========================================
plt.figure(figsize=(10, 6))

# Create plot
ax = sns.barplot(data=plot_data, x='Anomaly Type', y='Count', palette=plot_data['Color_Code'].tolist(), width=0.3) # Added width=0.3

# Remove unnecessary spines
sns.despine(left=True, bottom=True)

# Add large, bold number labels
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + (height*0.05), # Slightly above bar
            f'{int(height):,}', # Add comma for thousands
            ha='center', va='bottom',
            fontsize=28, fontweight='heavy', color=plot_data['Color_Code'][i])

# Clean up axes
plt.title('Weekend Anomalies: The Two Key Risks Found', fontsize=16, fontweight='bold', pad=20, loc='left')
plt.ylabel('')
plt.xlabel('')
plt.yticks([]) # Hide y-axis numbers as the labels do the job

plt.tight_layout()
plt.show()

Time-Series Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the data
df = pd.read_csv('cleaned_data_with_cashflow.csv', low_memory=False)

# 2. Split into Inflows and Outflows
inflows = df[df['Amount in USD'] > 0].copy()
outflows = df[df['Amount in USD'] < 0].copy()

# 3. Aggregate by "Client" (Document Header Text)
# Note: We treat Document Header Text as the Client Identifier
top_inflow_clients = inflows.groupby(['Document Header Text', 'Name'])['Amount in USD'].sum().nlargest(10).reset_index()
top_outflow_clients = outflows.groupby(['Document Header Text', 'Name'])['Amount in USD'].sum().abs().nlargest(10).reset_index()

# AstraZeneca Branding Colors
az_purple = '#830051'
az_green = '#C4D600'

# ==========================================
# VISUALIZATION 1: Top 10 Outcome "Clients" (Outflows)
# ==========================================
plt.figure(figsize=(12, 7))
sns.barplot(data=top_outflow_clients, x='Amount in USD', y='Document Header Text', hue='Name', dodge=False)
plt.title('Top 10 "Clients" by Total Outcome (Money Out)', fontsize=14)
plt.xlabel('Total Absolute Amount (USD)')
plt.ylabel('Client ID / Header Text')
plt.legend(title='Legal Entity')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# ==========================================
# VISUALIZATION 2: Top 10 Income "Clients" (Inflows)
# ==========================================
plt.figure(figsize=(12, 7))
plt.barplot_plot = sns.barplot(data=top_inflow_clients, x='Amount in USD', y='Document Header Text', hue='Name', dodge=False, palette='viridis')
plt.title('Top 10 "Clients" by Total Income (Money In)', fontsize=14)
plt.xlabel('Total Amount (USD)')
plt.ylabel('Client ID / Header Text')
plt.legend(title='Legal Entity')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# ==========================================
# DEEP DIVE: What are these clients doing? (Category Breakdown)
# ==========================================
# Let's look at the #1 Top Outflow Client
top_client_id = top_outflow_clients.iloc[0]['Document Header Text']
top_client_data = df[df['Document Header Text'] == top_client_id]

print(f"\n--- Deep Dive: Top Outflow Client ({top_client_id}) ---")
print(top_client_data.groupby('Category')['Amount in USD'].sum().abs().sort_values(ascending=False))

# Forecasting

Model Battle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# STEP 1: MODEL BATTLE (VALIDATION)
# Goal: Test 3 models on the last 4 weeks of KNOWN history
# ---------------------------------------------------------

# 1. Load & Clean Data
df = pd.read_csv('cleaned_data_with_cashflow.csv')
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

# Filter Duplicates
if 'is_duplicate' in df.columns:
    df_clean = df[df['is_duplicate'] == False].copy()
else:
    cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_clean = df.drop_duplicates(subset=cols, keep='first')

# Aggregate Weekly
weekly_df = df_clean.set_index('Pstng Date')['Amount in USD'].resample('W').sum().reset_index()
weekly_df.columns = ['Date', 'Net Cash Flow']

# 2. Train/Test Split
test_weeks = 4
train_series = weekly_df['Net Cash Flow'].iloc[:-test_weeks]
test_series = weekly_df['Net Cash Flow'].iloc[-test_weeks:]
test_dates = weekly_df['Date'].iloc[-test_weeks:]

# 3. Model A: Holt-Winters (Seasonal)
model_hw = ExponentialSmoothing(
    train_series,
    seasonal_periods=4,
    trend='add',
    seasonal='add',
    damped_trend=True
).fit()
preds_hw = model_hw.forecast(test_weeks)

# 4. Model B: ARIMA (Trend)
# model_arima = ARIMA(train_series, order=(1,1,1)).fit()
# preds_arima = model_arima.forecast(test_weeks)

# 5. Model C: Random Forest (Machine Learning)
# Feature Engineering
ml_df = weekly_df.copy()
ml_df['Lag_1'] = ml_df['Net Cash Flow'].shift(1)
ml_df['Lag_2'] = ml_df['Net Cash Flow'].shift(2)
ml_df['Lag_4'] = ml_df['Net Cash Flow'].shift(4)
ml_df['Rolling_Mean_4'] = ml_df['Net Cash Flow'].shift(1).rolling(window=4).mean()
ml_df = ml_df.dropna()

X = ml_df[['Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4']]
y = ml_df['Net Cash Flow']

# Split ML Data
X_train = X.iloc[:-test_weeks]
y_train = y.iloc[:-test_weeks]
X_test = X.iloc[-test_weeks:]
y_test = y.iloc[-test_weeks:]

# Train RF
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
preds_rf = model_rf.predict(X_test)

# 6. Evaluation Metrics
metrics = []
for name, preds in [('Holt-Winters', preds_hw), ('Random Forest', preds_rf)]:
    mae = mean_absolute_error(test_series, preds)
    rmse = np.sqrt(mean_squared_error(test_series, preds))
    metrics.append({'Model': name, 'MAE ($)': mae, 'RMSE ($)': rmse})

metrics_df = pd.DataFrame(metrics).sort_values(by='RMSE ($)')

# 7. Visualization
plt.figure(figsize=(12, 6))
# Plot actuals (last 12 weeks for context)
plt.plot(weekly_df['Date'].iloc[-12:], weekly_df['Net Cash Flow'].iloc[-12:], label='Actual Data', color='black', linewidth=2)
# Plot predictions
plt.plot(test_dates, preds_hw, label='Holt-Winters', marker='o', linestyle='--')
# plt.plot(test_dates, preds_arima, label='ARIMA', marker='x', linestyle='--')
plt.plot(test_dates, preds_rf, label='Random Forest', marker='s', linestyle='--')

plt.title('Step 1: Model Validation (Comparison on Last 4 Weeks)')
plt.ylabel('Net Cash Flow (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step1_model_comparison.png')

# Output
print("\n=== STEP 1: MODEL LEADERBOARD ===")
pd.options.display.float_format = '{:,.0f}'.format
print(metrics_df)

metrics_df.to_csv('step1_model_metrics.csv', index=False)
print("\nFiles Saved:")
print("- step1_model_comparison.png (Chart)")
print("- step1_model_metrics.csv (Table)")

1 month ahead forecast using Random Forest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# STEP 2: 1-MONTH FORECAST (OPERATIONAL VIEW)
# Recommended Model: Random Forest (Winner of Step 1)
# ---------------------------------------------------------

# 1. Load Data
df = pd.read_csv('cleaned_data_with_cashflow.csv')
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

# Filter Duplicates
if 'is_duplicate' in df.columns:
    df_clean = df[df['is_duplicate'] == False].copy()
else:
    cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_clean = df.drop_duplicates(subset=cols, keep='first')

# Aggregate Weekly (Full History)
full_weekly_df = df_clean.set_index('Pstng Date')['Amount in USD'].resample('W').sum().reset_index()
full_weekly_df.columns = ['Date', 'Net Cash Flow']

# 2. Feature Engineering (Prepare for RF)
ml_df = full_weekly_df.copy()
ml_df['Lag_1'] = ml_df['Net Cash Flow'].shift(1)
ml_df['Lag_2'] = ml_df['Net Cash Flow'].shift(2)
ml_df['Lag_4'] = ml_df['Net Cash Flow'].shift(4)
ml_df['Rolling_Mean_4'] = ml_df['Net Cash Flow'].shift(1).rolling(window=4).mean()
ml_df = ml_df.dropna()

# 3. Train on FULL History
X_train = ml_df[['Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4']]
y_train = ml_df['Net Cash Flow']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 4. Recursive Forecast (Next 4 Weeks)
forecast_horizon = 4
current_history = list(full_weekly_df['Net Cash Flow'].values)
rf_predictions = []

for i in range(forecast_horizon):
    # Build features dynamically
    feat_lag_1 = current_history[-1]
    feat_lag_2 = current_history[-2]
    feat_lag_4 = current_history[-4]
    feat_rolling = np.mean(current_history[-4:])

    # Predict
    features = np.array([[feat_lag_1, feat_lag_2, feat_lag_4, feat_rolling]])
    pred = rf_model.predict(features)[0]

    rf_predictions.append(pred)
    current_history.append(pred)

# 5. Create DataFrame
last_date = full_weekly_df['Date'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=forecast_horizon, freq='W')
df_forecast_1mo = pd.DataFrame({'Date': future_dates, 'Forecast': rf_predictions})

# 6. Visualization (Zoom In)
plt.figure(figsize=(10, 6))
# Show last 8 weeks of actuals + 4 weeks forecast
recent_history = full_weekly_df.iloc[-8:]
plt.plot(recent_history['Date'], recent_history['Net Cash Flow'], label='Recent Actuals', color='black', marker='.')
plt.plot(df_forecast_1mo['Date'], df_forecast_1mo['Forecast'], label='1-Month Forecast (Random Forest)', color='green', marker='s', linestyle='--')

plt.title('Step 2: Short-Term Liquidity Forecast (1 Month)')
plt.ylabel('Net Cash Flow (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step2_1month_forecast.png')

# Output
print("\n=== STEP 2: 1-MONTH FORECAST (RANDOM FOREST) ===")
pd.options.display.float_format = '{:,.2f}'.format
print(df_forecast_1mo)

df_forecast_1mo.to_csv('step2_1month_forecast.csv', index=False)
print("\nFiles Saved:")
print("- step2_1month_forecast.png")
print("- step2_1month_forecast.csv")

6 month Forecasting using Holt-Winters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ---------------------------------------------------------
# STEP 3: 6-MONTH FORECAST (STRATEGIC VIEW)
# Recommended Model: Holt-Winters (Stability & Seasonality)
# ---------------------------------------------------------

# 1. Load Data
df = pd.read_csv('cleaned_data_with_cashflow.csv')
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

# Filter Duplicates
if 'is_duplicate' in df.columns:
    df_clean = df[df['is_duplicate'] == False].copy()
else:
    cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_clean = df.drop_duplicates(subset=cols, keep='first')

# Aggregate Weekly (Full History)
weekly_df = df_clean.set_index('Pstng Date')['Amount in USD'].resample('W').sum().reset_index()
weekly_df.columns = ['Date', 'Net Cash Flow']

# 2. Train Holt-Winters on FULL History
# Configuration: Additive Trend, Additive Seasonality, Damped Trend (Prevents explosion)
model_hw = ExponentialSmoothing(
    weekly_df['Net Cash Flow'],
    seasonal_periods=4,
    trend='add',
    seasonal='add',
    damped_trend=True
).fit()

# 3. Forecast Next 26 Weeks (6 Months)
forecast_horizon = 26
forecast_values = model_hw.forecast(forecast_horizon)

# 4. Create DataFrame
last_date = weekly_df['Date'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=forecast_horizon, freq='W')

df_forecast_6mo = pd.DataFrame({
    'Date': future_dates,
    'Forecasted Net Cash Flow': forecast_values
})

# 5. Calculate Cumulative Cash Position (Relative to Today)
# Since we don't have the absolute bank balance file, we show the *change* in position
df_forecast_6mo['Cumulative Change'] = df_forecast_6mo['Forecasted Net Cash Flow'].cumsum()

# 6. Visualization (Zoom Out)
plt.figure(figsize=(12, 6))

# Plot History (Last 6 Months)
history_subset = weekly_df.iloc[-24:]
plt.plot(history_subset['Date'], history_subset['Net Cash Flow'], label='Historical Actuals', color='black', alpha=0.7)

# Plot Forecast
plt.plot(df_forecast_6mo['Date'], df_forecast_6mo['Forecasted Net Cash Flow'], label='6-Month Forecast (Holt-Winters)', color='blue', marker='o')

# Add a smoothed trend line to show the "Story"
z = np.polyfit(range(len(df_forecast_6mo)), df_forecast_6mo['Forecasted Net Cash Flow'], 1)
p = np.poly1d(z)
plt.plot(df_forecast_6mo['Date'], p(range(len(df_forecast_6mo))), "r--", alpha=0.5, label='Underlying Trend')

plt.title('Step 3: Strategic Liquidity Forecast (6 Months)')
plt.ylabel('Net Cash Flow (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step3_6month_forecast.png')

# Output
print("\n=== STEP 3: 6-MONTH FORECAST (HOLT-WINTERS) ===")
pd.options.display.float_format = '{:,.2f}'.format
print(df_forecast_6mo.head())
print(f"\nFinal Cumulative Change after 6 Months: ${df_forecast_6mo['Cumulative Change'].iloc[-1]:,.2f}")

df_forecast_6mo.to_csv('step3_6month_forecast.csv', index=False)
print("\nFiles Saved:")
print("- step3_6month_forecast.png")
print("- step3_6month_forecast.csv")

Forecasted vs Actual

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ---------------------------------------------------------
# GENERATE POWER BI READY FILE: "FORECAST VS ACTUALS"
# ---------------------------------------------------------

# 1. Load Data
df = pd.read_csv('cleaned_data_with_cashflow.csv')
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

# Filter Duplicates
if 'is_duplicate' in df.columns:
    df_clean = df[df['is_duplicate'] == False].copy()
else:
    cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_clean = df.drop_duplicates(subset=cols, keep='first')

# Aggregate Weekly
weekly_df = df_clean.set_index('Pstng Date')['Amount in USD'].resample('W').sum().reset_index()
weekly_df.columns = ['Date', 'Net Cash Flow']

# 2. Re-Run Model Validation (Last 4 Weeks)
test_weeks = 4
train_series = weekly_df['Net Cash Flow'].iloc[:-test_weeks]
test_series = weekly_df['Net Cash Flow'].iloc[-test_weeks:]
test_dates = weekly_df['Date'].iloc[-test_weeks:]

# A. Holt-Winters Predictions
hw_model = ExponentialSmoothing(train_series, seasonal_periods=4, trend='add', seasonal='add', damped_trend=True).fit()
hw_preds = hw_model.forecast(test_weeks)

# B. Random Forest Predictions
ml_df = weekly_df.copy()
ml_df['Lag_1'] = ml_df['Net Cash Flow'].shift(1)
ml_df['Lag_2'] = ml_df['Net Cash Flow'].shift(2)
ml_df['Lag_4'] = ml_df['Net Cash Flow'].shift(4)
ml_df['Rolling_Mean_4'] = ml_df['Net Cash Flow'].shift(1).rolling(window=4).mean()
ml_df = ml_df.dropna()

X = ml_df[['Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4']]
y = ml_df['Net Cash Flow']
X_train = X.iloc[:-test_weeks]
y_train = y.iloc[:-test_weeks]
X_test = X.iloc[-test_weeks:]
rf_model = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

# 3. Create the "Comparison DataFrame"
comparison_df = pd.DataFrame({
    'Date': test_dates,
    'Actual Net Cash Flow': test_series,
    'Forecast (Random Forest)': rf_preds,
    'Forecast (Holt-Winters)': hw_preds
})

# Calculate Errors for the CSV (so user can show them in tooltip)
comparison_df['Diff_RF'] = comparison_df['Forecast (Random Forest)'] - comparison_df['Actual Net Cash Flow']
comparison_df['Diff_HW'] = comparison_df['Forecast (Holt-Winters)'] - comparison_df['Actual Net Cash Flow']

# 4. Save
comparison_df.to_csv('powerbi_comparison_data.csv', index=False)

print("Generated 'powerbi_comparison_data.csv'")
print("Columns: Date, Actual Net Cash Flow, Forecast (Random Forest), Forecast (Holt-Winters), Diff_RF, Diff_HW")
print("\nPreview:")
print(comparison_df.head())

In [ ]:
import pandas as pd
import numpy as np

# 1. Load Data
main_df = pd.read_csv('cleaned_data_with_cashflow.csv') # Changed from 'Datathon Dataset.xlsx - Data - Main.csv'
main_df['Pstng Date'] = pd.to_datetime(main_df['Pstng Date'])

# 2. Strict Deduplication (Same as before)
dup_cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
main_df_filled = main_df.fillna({'Offst.acct': 'Unknown', 'Reference': 'Unknown', 'Assignment': 'Unknown'})
is_duplicate = main_df_filled.duplicated(subset=dup_cols, keep='first')
df_clean = main_df[~is_duplicate].copy()

# 3. Anomaly Detection (Z-Score Logic)
# Calculate Mean and Std Dev per Category
df_clean['Category_Mean'] = df_clean.groupby('Category')['Amount in USD'].transform('mean')
df_clean['Category_Std'] = df_clean.groupby('Category')['Amount in USD'].transform('std')

# Initialize Z-Score column
df_clean['Z_Score'] = 0.0

# Calculate Z-Score only where Std Dev > 0 (avoid division by zero)
mask = df_clean['Category_Std'] > 0
df_clean.loc[mask, 'Z_Score'] = (
    (df_clean.loc[mask, 'Amount in USD'] - df_clean.loc[mask, 'Category_Mean'])
    / df_clean.loc[mask, 'Category_Std']
)

# Identify Anomalies (Z-Score > 3 or < -3)
anomalies = df_clean[np.abs(df_clean['Z_Score']) > 3].copy()

# Sort by magnitude of Z-Score for impact
anomalies = anomalies.sort_values(by='Z_Score', key=abs, ascending=False)

# Select columns relevant for the Audit Report
cols_to_save = ['Pstng Date', 'Name', 'Category', 'Reference', 'Amount in USD', 'Z_Score', 'DocumentNo']
anomalies_final = anomalies[cols_to_save]

# 4. Save File
anomalies_final.to_csv('anomalies_transaction_level.csv', index=False)

print(f"Generated 'anomalies_transaction_level.csv' with {len(anomalies_final)} rows.")
print(anomalies_final.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ==========================================
# MASTER DASHBOARD DATA GENERATOR
# ==========================================
# This script creates 5 optimized CSV files for Power BI / Tableau

# Path to the Excel file (already mounted from previous steps)
path = '/content/drive/My Drive/Datathon Dataset.xlsx'

# 1. Load All Source Files
main_df = pd.read_csv('cleaned_data_with_cashflow.csv') # Use cleaned if available, or process Main
balance_df = pd.read_excel(path, sheet_name='Data - Cash Balance')
country_map = pd.read_excel(path, sheet_name='Others - Country Mapping')
cat_linkage = pd.read_excel(path, sheet_name='Others - Category Linkage')

# Pre-processing
main_df['Pstng Date'] = pd.to_datetime(main_df['Pstng Date'])

# Deduplication (Standard)
if 'is_duplicate' in main_df.columns:
    df_clean = main_df[main_df['is_duplicate'] == False].copy()
else:
    cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_clean = main_df.drop_duplicates(subset=cols, keep='first')

# ------------------------------------------------------------------
# TABLE 1: DASHBOARD_MASTER_LINE_CHART.csv
# (History + Short Term Forecast + Long Term Forecast)
# ------------------------------------------------------------------
# A. History
history = df_clean.set_index('Pstng Date')['Amount in USD'].resample('W').sum().reset_index()
history.columns = ['Date', 'Net Cash Flow']
history['Type'] = 'Historical Actuals'

# B. Forecasts (Re-generating briefly to ensure alignment)
# Setup
train_series = history['Net Cash Flow']
hw_model = ExponentialSmoothing(train_series, seasonal_periods=4, trend='add', seasonal='add', damped_trend=True).fit()
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Features for RF
ml_df = history.copy()
ml_df['Lag_1'] = ml_df['Net Cash Flow'].shift(1)
ml_df['Lag_2'] = ml_df['Net Cash Flow'].shift(2)
ml_df['Lag_4'] = ml_df['Net Cash Flow'].shift(4)
ml_df['Rolling_Mean_4'] = ml_df['Net Cash Flow'].shift(1).rolling(window=4).mean()
ml_df = ml_df.dropna()
rf_model.fit(ml_df[['Lag_1', 'Lag_2', 'Lag_4', 'Rolling_Mean_4']], ml_df['Net Cash Flow'])

# Generate Future Dates
last_date = history['Date'].iloc[-1]
future_dates_1mo = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=4, freq='W')
future_dates_6mo = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=26, freq='W')

# 1-Month RF Forecast
curr_hist = list(history['Net Cash Flow'].values)
rf_preds = []
for _ in range(4):
    feat = np.array([[curr_hist[-1], curr_hist[-2], curr_hist[-4], np.mean(curr_hist[-4:])]])
    pred = rf_model.predict(feat)[0]
    rf_preds.append(pred)
    curr_hist.append(pred)

df_rf = pd.DataFrame({'Date': future_dates_1mo, 'Net Cash Flow': rf_preds, 'Type': 'Short Term Forecast (ML)'})

# 6-Month HW Forecast
hw_preds = hw_model.forecast(26)
df_hw = pd.DataFrame({'Date': future_dates_6mo, 'Net Cash Flow': hw_preds, 'Type': 'Long Term Forecast (Strategy)'})

# C. Calculate Balances
global_start = balance_df['Carryforward Balance (USD)'].sum()
master_line = pd.concat([history, df_rf, df_hw])
# Calculate cumulative sum separately for History and Forecasts to connect them?
# Actually, for dashboard simplicity, just cumulative sum relative to start
# But forecasts need to branch off.
# Let's add 'Running Balance' only to History and HW (Strategy)
history['Running Balance'] = global_start + history['Net Cash Flow'].cumsum()
last_bal = history['Running Balance'].iloc[-1]
df_hw['Running Balance'] = last_bal + df_hw['Net Cash Flow'].cumsum()
df_rf['Running Balance'] = last_bal + df_rf['Net Cash Flow'].cumsum() # For short term view

master_chart = pd.concat([history, df_rf, df_hw])
master_chart.to_csv('dashboard_master_line_chart.csv', index=False)

# ------------------------------------------------------------------
# TABLE 2: DASHBOARD_WATERFALL_DRIVERS.csv
# (Aggregated by Category for Waterfall/Bar charts)
# ------------------------------------------------------------------
# Map Categories to Inflow/Outflow if not present
# The file 'Category Linkage' has this mapping
cat_map = dict(zip(cat_linkage['Category Names'], cat_linkage['Category'])) # Name -> Inflow/Outflow

df_clean['Flow Type'] = df_clean['Category'].map(cat_map).fillna('Unknown')

# Aggregation
waterfall = df_clean.groupby(['Category', 'Flow Type'])['Amount in USD'].sum().reset_index()
waterfall.columns = ['Category', 'Flow Type', 'Total Amount']
# Add Opening Balance Row
open_bal_row = pd.DataFrame([{'Category': 'Opening Balance', 'Flow Type': 'Balance', 'Total Amount': global_start}])
waterfall_final = pd.concat([open_bal_row, waterfall])
waterfall_final.to_csv('dashboard_waterfall_drivers.csv', index=False)

# ------------------------------------------------------------------
# TABLE 3: DASHBOARD_REGIONAL_MAP.csv
# (Entity Level Performance)
# ------------------------------------------------------------------
entity_agg = df_clean.groupby('Name')['Amount in USD'].sum().reset_index()
entity_agg.columns = ['Name', 'Net Cash Flow']
# Join with Balance
bal_agg = balance_df.groupby('Name')['Carryforward Balance (USD)'].sum().reset_index()
regional = pd.merge(entity_agg, bal_agg, on='Name', how='left').fillna(0)
regional['Ending Balance'] = regional['Carryforward Balance (USD)'] + regional['Net Cash Flow']
# Join with Country Map
regional = pd.merge(regional, country_map, left_on='Name', right_on='Code', how='left')
regional.to_csv('dashboard_regional_map.csv', index=False)

# ------------------------------------------------------------------
# TABLE 4: DASHBOARD_DAY_ANALYSIS.csv
# (Weekend vs Weekday Pattern)
# ------------------------------------------------------------------
df_clean['Day of Week'] = df_clean['Pstng Date'].dt.day_name()
day_agg = df_clean.groupby('Day of Week').agg(
    Transaction_Count=('Amount in USD', 'count'),
    Total_Volume=('Amount in USD', 'sum')
).reset_index()
# Sort days
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_agg['Day of Week'] = pd.Categorical(day_agg['Day of Week'], categories=days_order, ordered=True)
day_agg = day_agg.sort_values('Day of Week')
day_agg.to_csv('dashboard_day_analysis.csv', index=False)

print("Dashboard Data Generation Complete.")
print("Files Created:")
print("1. dashboard_master_line_chart.csv (For Pages 1 & 2)")
print("2. dashboard_waterfall_drivers.csv (For Page 3)")
print("3. dashboard_regional_map.csv (For Page 5)")
print("4. dashboard_day_analysis.csv (For Page 3 - Weekend Pattern)")
print("(Use 'anomalies_transaction_level.csv' from previous step for Page 4)")

# Display the first few rows of dashboard_master_line_chart.csv
df_master_line_chart = pd.read_csv('dashboard_master_line_chart.csv')
print("\nFirst 5 rows of dashboard_master_line_chart.csv:")
print(df_master_line_chart.head())

In [ ]:
import pandas as pd
import numpy as np

# 1. Load Data
# Use the cleaned_data_with_cashflow.csv generated earlier
df = pd.read_csv('cleaned_data_with_cashflow.csv', low_memory=False)
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'])

# 2. Strict Deduplication (Crucial step to remove noise)
cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
df_clean = df.drop_duplicates(subset=cols, keep='first')

# 3. Aggregate to Weekly Net Cash Flow
# We use 'W' (Weekly) because daily data is naturally noisy (zero on weekends)
weekly_data = df_clean.set_index('Pstng Date')['Amount in USD'].resample('W').sum()

# 4. Calculate Coefficient of Variation (CV)
# Formula: Standard Deviation / Absolute Mean
mean_val = weekly_data.mean()
std_val = weekly_data.std()

volatility_score = std_val / abs(mean_val)

print(f"--- Volatility Calculation ---")
print(f"Mean Weekly Cash Flow: ${mean_val:,.2f}")
print(f"Standard Deviation:    ${std_val:,.2f}")
print(f"Volatility Score (CV): {volatility_score:.2f}")

if volatility_score > 1.0:
    print("Result: HIGH VOLATILITY. (Standard Linear Models will likely fail).")
else:
    print("Result: Low Volatility. (Linear trends might work).")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import os # Import os for file path checks
from google.colab import drive # Import drive for mounting

# Define the path for the cleaned data file
cleaned_file_path = 'cleaned_data_with_cashflow.csv'

# 1. Load Your Cleaned Data (Robustly)
# -------------------------------------------------------
if not os.path.exists(cleaned_file_path):
    print(f"'{cleaned_file_path}' not found. Performing initial data loading and cleaning...")

    # Check if drive is already mounted to avoid duplicate messages
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Prioritize mounted drive path, fallback to local path if not found
    excel_path_drive = '/content/drive/My Drive/Datathon Dataset.xlsx'
    excel_path_local = '/content/Datathon Dataset.xlsx' # Assuming it might be directly uploaded

    if os.path.exists(excel_path_drive):
        path = excel_path_drive
    elif os.path.exists(excel_path_local):
        path = excel_path_local
    else:
        raise FileNotFoundError(f"Datathon Dataset.xlsx not found at {excel_path_drive} or {excel_path_local}")

    # Read main data from the 'Data - Main' sheet
    df_main_init = pd.read_excel(path, sheet_name='Data - Main')

    # --- DROP EMPTY COLUMNS --- (from D5uhZJbaVVox)
    cols_to_drop = ['Cost Ctr', 'WBS element', 'Ref.key (header) 1', 'Clrng doc.']
    df_main_init = df_main_init.drop(columns=cols_to_drop, errors='ignore')

    # --- DROP SPECIFIC COLUMNS BY INDEX (corresponding to names) --- (from D5uhZJbaVVox)
    columns_to_drop_by_name = ['Amount in doc. curr.', 'Amt in loc.cur.', 'Curr.', 'LCurr', 'Rate (USD)']
    df_main_init = df_main_init.drop(columns=columns_to_drop_by_name, errors='ignore')

    # --- CUSTOM CLEANING: UPDATE 'Non Netting AP' category and index --- (from D5uhZJbaVVox)
    df_main_init['Category'] = df_main_init['Category'].replace({'Non Netting AP': 'Non-Netting AP'})
    df_main_init.loc[df_main_init['Category'] == 'Non-Netting AP', 'Category Index'] = 6

    # --- ANOMALY FLAGGING & CASHFLOW BUCKET (from HNBiu2DO0gVu) ---
    df_main_init['Pstng Date'] = pd.to_datetime(df_main_init['Pstng Date'], dayfirst=True) # Added dayfirst=True
    df_main_init['is_weekend'] = df_main_init['Pstng Date'].dt.dayofweek >= 5
    duplicate_cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    df_main_init['is_duplicate'] = df_main_init.duplicated(subset=duplicate_cols, keep='first')
    df_main_init['z_score'] = df_main_init.groupby('Category')['Amount in USD'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    df_main_init['is_outlier'] = np.abs(df_main_init['z_score']) > 3
    df_main_init = df_main_init.drop(columns=['z_score'])
    financing_categories = [12, 13, 14, 15, 19, 20, 21, 24]
    df_main_init['cashflow_Bucket'] = 'Operating'
    df_main_init.loc[df_main_init['Category Index'].isin(financing_categories), 'cashflow_Bucket'] = 'Financing'

    # Save the cleaned data for future use
    df_main_init.to_csv(cleaned_file_path, index=False)
    print(f"Successfully created and saved '{cleaned_file_path}'.")

df = pd.read_csv(cleaned_file_path, low_memory=False)
df['Pstng Date'] = pd.to_datetime(df['Pstng Date'], dayfirst=True) # Added dayfirst=True

# Filter duplicates to ensure accuracy
if 'is_duplicate' in df.columns:
    df = df[df['is_duplicate'] == False]
else: # If 'is_duplicate' column doesn't exist (e.g., if re-running part of the code)
    cols = ['Name', 'Pstng Date', 'Amount in USD', 'Offst.acct', 'Reference', 'Assignment']
    # Fill NaN to treat them as distinct for duplication check
    df_filled_for_dup = df.fillna({'Offst.acct': 'NULL', 'Reference': 'NULL', 'Assignment': 'NULL'})
    df = df[~df_filled_for_dup.duplicated(subset=cols, keep='first')]

# 2. Prepare History (Jan - Oct 2025)
# -------------------------------------------------------
# Aggregate to Weekly Net Flow
history_df = df.set_index('Pstng Date')['Amount in USD'].resample('W').sum().reset_index()
history_df.columns = ['Date', 'Net Cash Flow']

# Calculate Running Balance (Solvency)
# We assume a starting balance or start from 0 relative change
# (Adjust '9654990' if you want the exact global start balance from your file)
start_balance = 9654990 # This initial balance might come from 'Data - Cash Balance' sheet
history_df['Ending Balance'] = start_balance + history_df['Net Cash Flow'].cumsum()

# 3. Generate 6-Month Forecast (Holt-Winters)
# -------------------------------------------------------
model = ExponentialSmoothing(
    history_df['Net Cash Flow'],
    seasonal_periods=4,
    trend='add',
    seasonal='add',
    damped_trend=True
).fit()

forecast_horizon = 26 # 6 Months
forecast_values = model.forecast(forecast_horizon)

# Create Future DataFrame
last_date = history_df['Date'].iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=forecast_horizon, freq='W')
forecast_df = pd.DataFrame({'Date': future_dates, 'Net Cash Flow': forecast_values})

# Calculate Future Cumulative Balance
last_hist_bal = history_df['Ending Balance'].iloc[-1]
forecast_df['Ending Balance'] = last_hist_bal + forecast_df['Net Cash Flow'].cumsum()

# 4. PLOT THE CHART
# -------------------------------------------------------
plt.figure(figsize=(12, 7))

# A. Plot History (Solid Dark Blue)
plt.plot(history_df['Date'], history_df['Ending Balance'] / 1e6,
         label='Historical Actuals', color='#2c3e50', linewidth=2.5)

# B. Plot Forecast (Dotted Red - The "Burn Line")
plt.plot(forecast_df['Date'], forecast_df['Ending Balance'] / 1e6,
         label='Forecast Trend', color='#c0392b', linestyle='--', linewidth=3)

# C. Add Zero Line (Solvency Threshold)
plt.axhline(0, color='black', linewidth=1, linestyle='-', alpha=0.5)
plt.text(history_df['Date'].iloc[0], 0.5, ' Solvency Threshold ($0)', color='black', fontsize=10, va='bottom', fontweight='bold')

# D. ANNOTATION 1: THE NOV 23 SHOCK (-$4.5M)
# Find the dip in Nov
nov_mask = (forecast_df['Date'] >= '2025-11-01') & (forecast_df['Date'] <= '2025-11-30')
nov_data = forecast_df[nov_mask]
# Check if nov_data is empty before trying to access idxmin()
if not nov_data.empty:
    shock_point = nov_data.loc[nov_data['Net Cash Flow'].idxmin()] # Find biggest outflow week
    shock_date = shock_point['Date']
    shock_val = forecast_df.loc[forecast_df['Date'] == shock_date, 'Ending Balance'].values[0] / 1e6

    # Draw the Dot
    plt.scatter([shock_date], [shock_val], color='#e74c3c', s=150, zorder=10, edgecolor='white', linewidth=2)
    # Draw the Tag
    plt.annotate(f'IMMEDIATE SHOCK\nNov 23 Alert',
                 xy=(shock_date, shock_val),
                 xytext=(shock_date, shock_val + 15),
                 arrowprops=dict(facecolor='#e74c3c', shrink=0.05, width=2),
                 fontsize=11, fontweight='bold', color='white',
                 bbox=dict(boxstyle="round,pad=0.4", fc="#e74c3c", ec="none"))
else:
    print("No November data found for shock annotation.")

# E. ANNOTATION 2: THE STRUCTURAL BURN (-$64M)
# The last point
burn_point = forecast_df.iloc[-1]
burn_date = burn_point['Date']
burn_val = burn_point['Ending Balance'] / 1e6

# Draw the Dot
plt.scatter([burn_date], [burn_val], color='#c0392b', s=150, zorder=10, edgecolor='white', linewidth=2)
# Draw the Tag
plt.annotate(f'STRUCTURAL BURN\n${burn_val:.0f}M (April 2026)',
             xy=(burn_date, burn_val),
             xytext=(burn_date - pd.Timedelta(weeks=6), burn_val + 15),
             arrowprops=dict(facecolor='#c0392b', shrink=0.05, width=2),
             fontsize=12, fontweight='bold', color='white',
             bbox=dict(boxstyle="round,pad=0.4", fc="#c0392b", ec="none"))

# Styling
plt.title('The Financial Verdict: Solvency Risk Forecast (2025-2026)', fontsize=16, fontweight='bold', pad=20, color='#2c3e50')
plt.ylabel('Cash Balance (Millions USD)', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='upper right', frameon=True)

# Date Formatting
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.savefig('Slide8_Financial_Verdict.png', dpi=300)
print("Chart generated: Slide8_Financial_Verdict.png")
plt.show()